In [12]:
import fitz 
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    texts = []
    for page in doc:
        text = page.get_text()
        if text.strip(): 
            texts.append(text)
    return texts

pdf1_texts = extract_text_from_pdf(r"C:\Users\91936\Desktop\RAG\Medlink\konwledge\Concept-health-Rai-2016.pdf")
pdf2_texts = extract_text_from_pdf(r"C:\Users\91936\Desktop\RAG\Medlink\konwledge\Raven_Johnson_McGraw-Hill_Biology.pdf")

all_texts = pdf1_texts + pdf2_texts

print(f"Loaded {len(all_texts)} pages from both PDFs.")


MuPDF error: format error: cmsOpenProfileFromMem failed

MuPDF error: format error: cmsOpenProfileFromMem failed

MuPDF error: format error: cmsOpenProfileFromMem failed

MuPDF error: format error: cmsOpenProfileFromMem failed

MuPDF error: format error: cmsOpenProfileFromMem failed

MuPDF error: format error: cmsOpenProfileFromMem failed

MuPDF error: format error: cmsOpenProfileFromMem failed

MuPDF error: format error: cmsOpenProfileFromMem failed

MuPDF error: format error: cmsOpenProfileFromMem failed

MuPDF error: format error: cmsOpenProfileFromMem failed

Loaded 1285 pages from both PDFs.


In [13]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,        
    chunk_overlap=50,      
    separators=["\n\n", "\n", ".", " "]  
)

chunks = []
for page_text in all_texts:
    split_chunks = splitter.split_text(page_text)
    chunks.extend(split_chunks)

print(f"Total chunks created: {len(chunks)}")
print("Example chunk:\n", chunks[1111][:500])

Total chunks created: 16144
Example chunk:
 The first lens focuses the image of the object on the second
lens, which magnifies it again and focuses it on the back of
the eye. Microscopes that magnify in stages using several
lenses are called compound microscopes. They can resolve
structures that are separated by more than 200 nm. An image


In [14]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "intfloat/e5-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

def embed_chunk_multi_vector(text):
    encoded_input = tokenizer("passage: " + text, return_tensors="pt", truncation=True, padding=True, max_length=256).to(device)
    with torch.no_grad():
        output = model(**encoded_input)
    return output.last_hidden_state.squeeze(0).cpu() 


In [15]:
embedded_chunks = []

for i, chunk in enumerate(chunks[:500]):
    try:
        vec = embed_chunk_multi_vector(chunk)
        embedded_chunks.append({
            'text': chunk,
            'embedding': vec
        })
    except Exception as e:
        print(f"Error on chunk {i}: {e}")


In [16]:
def multi_vector_search(query, embedded_chunks, top_k=5):
    query_vecs = embed_chunk_multi_vector(query).to(torch.float32)  
    scored_chunks = []
    for item in embedded_chunks:
        doc_vecs = item['embedding'].to(torch.float32)  

        sim_matrix = torch.matmul(query_vecs, doc_vecs.T)

        score = sim_matrix.sum().item()

        scored_chunks.append((item['text'], score))

    scored_chunks.sort(key=lambda x: x[1], reverse=True)

    return scored_chunks[:top_k]


In [17]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=device)

def rerank_chunks(query, retrieved_chunks, top_k=3):
    pairs = [(query, chunk_text) for chunk_text, _ in retrieved_chunks]

    scores = reranker.predict(pairs)

    reranked = list(zip(pairs, scores))
    reranked.sort(key=lambda x: x[1], reverse=True)

    return reranked[:top_k]



In [18]:
query = "What is the function of red blood cells?"

top_chunks = multi_vector_search(query, embedded_chunks, top_k=5)

top_reranked = rerank_chunks(query, top_chunks, top_k=3)

for i, ((q, chunk), score) in enumerate(top_reranked):
    print(f"\nReranked Top {i+1} (score: {score:.2f}):\n{chunk}")



Reranked Top 1 (score: -10.63):
(CO2) and H2O join to form carbonic acid (H2CO3), which
in a second reaction dissociates to yield bicarbonate ion
(HCO3–) and H+ (figure 2.20). If some acid or other sub-
stance adds H+ ions to the blood, the HCO3– ions act as a
base and remove the excess H+ ions by forming H2CO3.

Reranked Top 2 (score: -11.16):
pH equals the exponent times –1. Thus, pure water, with an
[H+] of 10–7 mole/liter, has a pH of 7. Recall that for every
H+ ion formed when water dissociates, an OH– ion is also
formed, meaning that the dissociation of water produces H+
and OH– in equal amounts. Therefore, a pH value of 7 indi-

Reranked Top 3 (score: -11.29):
V
Cr
Cu
N
B
Co
Zn
Se
Mo
Sn
I 
8
14
13
26
20
11
19
12
1
25
9
15
6
16
17
23
24
29
7
5
27
30
34
42
50
53 
46.6
27.7
6.5
5.0
3.6
2.8
2.6
2.1
0.14
0.1
0.07
0.07
0.03
0.03
0.01
0.01
0.01
0.01
Trace
Trace
Trace
Trace
Trace
Trace
Trace
Trace 
65.0
Trace
Trace
Trace
1.5
0.2
0.4
0.1
9.5
Trace
Trace
1.0
18.5


In [19]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

generator_tokenizer = T5Tokenizer.from_pretrained("t5-base")
generator_model = T5ForConditionalGeneration.from_pretrained("t5-base").to(device)

def generate_answer(query, reranked_chunks, max_context_length=512):
    # Join top chunks as context
    context = " ".join([chunk for (q, chunk), score in reranked_chunks])
    
    # Format the prompt
    prompt = f"question: {query} context: {context}"

    # Tokenize input
    inputs = generator_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_context_length).to(device)

    # Generate output
    with torch.no_grad():
        output = generator_model.generate(inputs["input_ids"], max_length=100)

    # Decode the output
    return generator_tokenizer.decode(output[0], skip_special_tokens=True)


In [20]:
query = "What is the function of red blood cells?"

# Retrieve and rerank
retrieved = multi_vector_search(query, embedded_chunks, top_k=5)
reranked = rerank_chunks(query, retrieved, top_k=3)

# Generate answer
final_answer = generate_answer(query, reranked)

print(f"\nAnswer:\n{final_answer}")


Answer:
remove the excess H+ ions by forming H2CO3
